# 04 — Customer Segmentation
### Credit Card Customer Intelligence & Churn Analytics

**Goal:** Turn the EDA findings into business-actionable customer segments — no clustering algorithm needed. Rule-based segmentation is easier to explain to a business stakeholder and directly traceable to specific thresholds, which matters more here than marginal accuracy gains from k-means.

**Segments (per the brief), assigned by priority so each customer lands in exactly one bucket:**
1. **Dormant** — very low transaction activity + inactive + low revenue (address first: highest risk of full disengagement)
2. **At Risk** — high inactivity, low transactions, high utilization, high contacts (churn warning signs from EDA)
3. **New Customer** — low tenure (too early to classify by behavior yet)
4. **Premium** — high income + Platinum/Gold card + large credit limit
5. **High Value** — high transactions, high credit limit, low utilization, not churned
6. **Standard** — everyone else

Priority order matters: e.g. a high-income customer who's also gone inactive should be flagged **At Risk**, not filed under Premium and missed.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
PLOT_DIR = '../images/plots'

df = pd.read_csv('../data/processed/bankchurners_clean.csv')
print(df.shape)

(10127, 27)


## 1. Define thresholds

Using quantiles from the data itself rather than arbitrary numbers, so thresholds adapt to this customer base.

In [2]:
trans_ct_25 = df['Total_Trans_Ct'].quantile(0.25)
trans_amt_75 = df['Total_Trans_Amt'].quantile(0.75)
credit_limit_75 = df['Credit_Limit'].quantile(0.75)
util_75 = df['Avg_Utilization_Ratio'].quantile(0.75)
util_25 = df['Avg_Utilization_Ratio'].quantile(0.25)
tenure_25 = df['Months_on_book'].quantile(0.25)

print(f"Total_Trans_Ct 25th pct:   {trans_ct_25:.1f}")
print(f"Total_Trans_Amt 75th pct:  {trans_amt_75:.1f}")
print(f"Credit_Limit 75th pct:     {credit_limit_75:.1f}")
print(f"Utilization 75th pct:      {util_75:.3f}")
print(f"Utilization 25th pct:      {util_25:.3f}")
print(f"Months_on_book 25th pct:   {tenure_25:.1f}")

Total_Trans_Ct 25th pct:   45.0
Total_Trans_Amt 75th pct:  4741.0
Credit_Limit 75th pct:     11067.5
Utilization 75th pct:      0.503
Utilization 25th pct:      0.023
Months_on_book 25th pct:   31.0


## 2. Assign segments by priority

In [3]:
def assign_segment(row):
    # 1. Dormant — very low activity, inactive, low balance
    if (row['Total_Trans_Ct'] <= trans_ct_25 and
        row['Months_Inactive_12_mon'] >= 3 and
        row['Total_Revolving_Bal'] < df['Total_Revolving_Bal'].median()):
        return 'Dormant'

    # 2. At Risk — inactive, low transactions, high utilization, high contacts
    if (row['Months_Inactive_12_mon'] >= 3 and
        row['Total_Trans_Ct'] <= trans_ct_25 and
        (row['Avg_Utilization_Ratio'] >= util_75 or row['Contacts_Count_12_mon'] >= 4)):
        return 'At Risk'

    # 3. New Customer — low tenure, not enough history to classify by behavior
    if row['Months_on_book'] <= tenure_25:
        return 'New Customer'

    # 4. Premium — high income, premium card, large limit
    if (row['Income_Category'] in ['$120K +', '$80K - $120K'] and
        row['Card_Category'] in ['Gold', 'Platinum'] and
        row['Credit_Limit'] >= credit_limit_75):
        return 'Premium'

    # 5. High Value — high transactions, high limit, low utilization, retained
    if (row['Total_Trans_Amt'] >= trans_amt_75 and
        row['Credit_Limit'] >= credit_limit_75 and
        row['Avg_Utilization_Ratio'] <= util_25 and
        row['Attrition_Flag'] == 'Existing Customer'):
        return 'High Value'

    return 'Standard'

df['Customer_Segment'] = df.apply(assign_segment, axis=1)
df['Customer_Segment'].value_counts()

Customer_Segment
Standard        6709
New Customer    2351
Dormant          679
At Risk          289
High Value        61
Premium           38
Name: count, dtype: int64

**Note on segment sizes:** Premium and High Value are intentionally narrow (they require multiple simultaneous high-bar conditions) — that's expected for the small population that truly represents the bank's most valuable customers. If any segment came out empty, thresholds would need loosening; that isn't the case here.

## 3. Segment size & churn rate

In [4]:
segment_summary = df.groupby('Customer_Segment').agg(
    Customer_Count=('Churn_Flag', 'size'),
    Churn_Rate=('Churn_Flag', 'mean'),
    Avg_Credit_Limit=('Credit_Limit', 'mean'),
    Avg_Trans_Amt=('Total_Trans_Amt', 'mean'),
    Avg_Trans_Ct=('Total_Trans_Ct', 'mean'),
    Avg_Utilization=('Avg_Utilization_Ratio', 'mean'),
    Avg_Revolving_Bal=('Total_Revolving_Bal', 'mean'),
    Avg_Engagement=('Engagement_Score', 'mean'),
).sort_values('Customer_Count', ascending=False)

segment_summary['Pct_of_Base'] = (segment_summary['Customer_Count'] / len(df) * 100).round(1)
segment_summary.round(2)

,Customer_Count,Churn_Rate,Avg_Credit_Limit,Avg_Trans_Amt,Avg_Trans_Ct,Avg_Utilization,Avg_Revolving_Bal,Avg_Engagement,Pct_of_Base
Customer_Segment,,,,,,,,,
Standard,6709,0.12,8762.77,4631.07,67.78,0.28,1211.24,56.46,66.2
New Customer,2351,0.12,8321.92,4631.79,67.55,0.29,1184.49,57.78,23.2
Dormant,679,0.65,7512.04,1835.07,35.51,0.10,360.07,38.82,6.7
At Risk,289,0.30,4413.83,1726.98,35.09,0.63,1961.45,42.65,2.9
High Value,61,0.00,22839.64,9943.31,96.44,0.00,57.05,53.20,0.6
Premium,38,0.13,34000.79,7613.26,82.97,0.04,1317.61,53.43,0.4


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

segment_summary['Customer_Count'].sort_values().plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title('Segment Size (# customers)')
axes[0].set_xlabel('Customers')

segment_summary['Churn_Rate'].sort_values().plot(kind='barh', ax=axes[1], color='#C44E52')
axes[1].set_title('Churn Rate by Segment')
axes[1].set_xlabel('Churn Rate')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/segment_size_and_churn.png', bbox_inches='tight')
plt.show()

## 4. Segment profiles — what makes each one distinct

In [6]:
profile_cols = ['Total_Trans_Amt','Total_Trans_Ct','Credit_Limit','Avg_Utilization_Ratio',
                'Months_Inactive_12_mon','Contacts_Count_12_mon','Engagement_Score']

fig, ax = plt.subplots(figsize=(11, 6))
# Normalize each column 0-1 across segment means, for a comparable radar-style view via heatmap instead (simpler, avoids radar-chart distortion)
profile_means = df.groupby('Customer_Segment')[profile_cols].mean()
profile_norm = (profile_means - profile_means.min()) / (profile_means.max() - profile_means.min())
sns.heatmap(profile_norm, annot=profile_means.round(1), fmt='g', cmap='YlGnBu', ax=ax, cbar_kws={'label':'Relative level (0-1)'})
ax.set_title('Segment Profile Heatmap (annotated with actual mean values)')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/segment_profile_heatmap.png', bbox_inches='tight')
plt.show()

## 5. Business read on each segment

- **Dormant** — lowest activity, already checked out in practice even if the account is technically still open. Highest priority for reactivation offers or a "we miss you" campaign; if no response, treat as a near-certain future churn.
- **At Risk** — still transacting somewhat but showing the exact warning signs the EDA flagged (inactivity + high utilization/contacts). This is the group where proactive outreach has the best chance of actually changing the outcome, since they haven't fully disengaged yet.
- **New Customer** — too early to judge; the priority here is onboarding completeness (are they using all the features/products available) rather than retention offers.
- **Premium** — small, high-income, premium-card holders. Retention matters disproportionately here since each customer represents high potential lifetime value — worth white-glove service and relationship-manager style attention rather than mass campaigns.
- **High Value** — the bank's best customers by the classic value criteria (spend, limit, low risk) who haven't churned. Priority is protecting the relationship (perks, limit increases) rather than fixing anything.
- **Standard** — the largest bucket by construction; average-behavior customers who don't trip any specific rule. Good candidates for standard engagement programs rather than targeted intervention.

## 6. Save segmented dataset

In [7]:
df.to_csv('../data/processed/bankchurners_segmented.csv', index=False)
segment_summary.to_csv('../data/processed/segment_summary.csv')
print("Saved bankchurners_segmented.csv and segment_summary.csv")

Saved bankchurners_segmented.csv and segment_summary.csv
